# Semantic correspondence: Colab runner

This notebook only sets options and calls the repo code (`scripts/colab_setup.py`, `run.py`).
Code changes belong in the repo: edits made here are lost. Full guide: `RUN.md`.

**Before the first run:**
1. Runtime > Change runtime type > **GPU** (needed for `train`; `evaluate`/`submit` also work on CPU, slowly).
2. Accept the competition rules on Kaggle.
3. Add these to Colab Secrets (key icon) and allow notebook access: `KAGGLE_USERNAME`, `KAGGLE_KEY`, `WANDB_API_KEY`, `HF_TOKEN`.

In [24]:
# for charge the .env file as the project .env.example required
import os
from dotenv import load_dotenv

# Path to the .env file (supports both /content/.env and repository root .env)
env_path = '/content/.env' if os.path.exists('/content/.env') else '.env'

# Load the environment variables
if os.path.exists(env_path):
    load_dotenv(dotenv_path=env_path, override=True)
    print(f"Loaded environment variables from {env_path}")
else:
    print(f"Warning: {env_path} not found. Please make sure the file exists.")

In [25]:
import json
from pathlib import Path

cfg_file = Path("/content/drive/MyDrive/aml_semantic_correspondence/outputs/20261120-203639_train/config.json")
cfg = json.loads(cfg_file.read_text())
cfg.update({
    "constellation_align": True,
    "flip_test": True,
    "soft_window": 7,
    "soft_temperature": 0.04,
})
cfg_file.write_text(json.dumps(cfg, indent=2) + "\n")

4525

In [41]:
# Run settings
BRANCH = "giuseppe"      # branch to run; push your changes there first
MODE = "train"
RUN = ""
RESUME = ""           # train: continue this run id after a disconnect
EXTRA = ""         # other run.py flags, e.g. "--dry-run"
PRESET = "unified-lora"

# Transformation switches (override BASE in src/config.py for this Colab session):
OBJECT_ZOOM = True
CROSS_TRANSFORM = False      # File 1: src/models/cross_transform.py (multi-scale + cross-image attention adapter)
CONSTELLATION_ALIGN = False # File 2: src/models/constellation.py (pair-level Sim(2) constellation prior at inference)
FUSION = "none"             # "none" (fast pure DINOv2) | "concat" | "adapter"
EVAL_IMAGE_SIZE = None

In [42]:
import os
from google.colab import userdata

token = userdata.get("GITHUB_TOKEN")
REPO_URL = f"https://{token}@github.com/julpfi/sapienza_aml_semantic_correspondence.git"
REPO_DIR = "/content/sapienza_aml_semantic_correspondence"
if not os.path.isdir(f"{REPO_DIR}/.git"):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
%run scripts/colab_setup.py --branch {BRANCH}

/content/sapienza_aml_semantic_correspondence
$ git fetch origin giuseppe
$ git checkout giuseppe
$ git pull --ff-only origin giuseppe


CalledProcessError: Command '['git', 'pull', '--ff-only', 'origin', 'giuseppe']' returned non-zero exit status 1.

In [36]:
# Apply the transformation switches above to src/config.py before running run.py
import re
from pathlib import Path

cfg_path = Path("src/config.py")
text = cfg_path.read_text()
text = re.sub(r'("cross_transform":\s*)(True|False)', rf'\g<1>{CROSS_TRANSFORM}', text)
text = re.sub(r'("constellation_align":\s*)(True|False)', rf'\g<1>{CONSTELLATION_ALIGN}', text)
text = re.sub(r'("fusion":\s*)"[^"]+"', rf'\g<1>"{FUSION}"', text)
cfg_path.write_text(text)
!grep -nE '"cross_transform":|"constellation_align":|"fusion":' src/config.py

200:    "fusion": "none",                  # "none" (default on giuseppe branch) | "concat" | "adapter"
214:    "cross_transform": False,           # True = train & apply CrossImageTransform; False = off
222:    "constellation_align": False,      # True = refine predictions with the pair's Sim(2) constellation prior
287:    "pair_batch": False, "hard_neg_weight": 1.0, "ema_decay": 0.0, "fusion": "none",
288:    "cross_transform": False, "constellation_align": False, "object_zoom": False,
298:    "aug_grayscale_prob": 0.1, "aug_crop_scale": [0.7, 1.0], "aug_blur_prob": 0.1, "fusion": "none",
299:    "cross_transform": False, "constellation_align": False, "object_zoom": False,


In [37]:
# Start the run. Runs are saved on your Drive: aml_semantic_correspondence/outputs/<run_id>
flags = f"--mode {MODE} {EXTRA}"
if RESUME:
    flags += f" --resume {RESUME}"
elif RUN:
    flags += f" --run {RUN}"
if MODE == "train":
    flags += " --require-gpu"
!python run.py {flags}

Note: run 20261121-164729_train started on 1ae61ee-dirty, code is now 0016531-dirty.
Run 20261121-164729_train -> /content/drive/MyDrive/aml_semantic_correspondence/outputs/20261121-164729_train (settings in config.json)

Loaded weights from /content/drive/MyDrive/aml_semantic_correspondence/outputs/20261121-164729_train/checkpoints/best.pt
/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (

feature cache: 1279 hits / 852 misses
wrote 577 predictions -> /content/drive/MyDrive/aml_semantic_correspondence/outputs/20261121-164729_train/predictions.csv
wrote 577 rows -> /content/drive/MyDrive/aml_semantic_correspondence/outputs/20261121-164729_train/submission.csv
{
  "columns_ok": true,
  "same_ids_and_order": true,
  "unique_ids": true,
  "all_finite": true,
  "n_rows": 577
}


In [19]:
# Optional: paired comparison of evaluated runs (seeds comma-separated, A then B), e.g. "a_s0,a_s1 b_s0,b_s1".
# Each run needs per_query.csv: MODE = "evaluate" with RUN = "<run_id>" writes it.
COMPARE = ""
if COMPARE:
    !python scripts/compare_runs.py {COMPARE}

In [38]:
# Optional: upload a submission to Kaggle (set the run id printed above).
SUBMIT_RUN = "20261121-164729_train"  # set to the run id you want to submit, or "" to skip
if SUBMIT_RUN:
    !python scripts/kaggle_io.py submit --file /content/drive/MyDrive/aml_semantic_correspondence/outputs/{SUBMIT_RUN}/submission.csv --message "{SUBMIT_RUN}"


Submission ref: 58680701
3 submissions remaining today.
Successfully submitted to AML Challenge 2026/27Submitted. Check the score: kaggle competitions submissions -c aml-challenge-2026-27
